# CodeSwitch-TTS — Plan 013 residual-only v3 run (standalone Kaggle notebook)

This notebook is the complete run sheet for the B/C noise fix after the v2
perceptual check:

1. sync the code from GitHub;
2. install the pinned dependency stack;
3. restore the latest student checkpoint and the cached 2 000-shard teacher cache from W&B;
4. run the pre-retrain code/audio diagnostic;
5. **residual-only retrain from v2** — no teacher pass, no draft pass; the
   residual is capacity-bumped (512-wide, 6 blocks), ramps to 100 % draft CB0
   over 100 of 150 epochs, and is selected on draft-CB0 validation loss;
6. push the v3 student to W&B as a recovery point;
7. run the post-retrain diagnostic and enforce `rt_a >= 5 %`, `rd_a >= 5 %`;
7b. if the gate fails, run the 10b draft-prior sweep — it is a
    diagnostic only and never retrains;
8. synthesize **one** held-out sentence in `argmax` mode, print its top-1
   code accuracy and DAC provenance, and save
   `output_audio/test_fixed_argmax.wav` for a listening check;
9. **stop** until that WAV is confirmed intelligible;
10. only then evaluate A/B/C in `argmax` mode;
11. evaluate B/C in matched `sample` mode;
12. enforce the CER/UTMOS/ASR/latency gate;
13. zip and back up the whole run to W&B.

## Before you click Run All

* **Push the branch first.** The default `REPO_REF` is `fix/013-bc-noise`. If the
  branch has been merged, change it to `main`. The Kaggle clone cannot see local
  commits.
* Kaggle **Add-ons → Secrets**:
  * `WANDB_API_KEY` — **required** (for both the teacher/student artifacts and the backup);
  * `HF_TOKEN` — optional; the models are public.
* Notebook settings: **GPU T4 x1**, **Internet ON**.
* The existing W&B `teacher` artifact must contain the 2 000 shards. This
  notebook deliberately refuses to start training on fewer than 1 500 shards,
  because that usually means you restored the 300-sentence dry-run cache.
* The pre-retrain diagnostic needs the latest student artifact. The notebook
  restores it before training. `DO_TRAIN` is **False** by default; the primary
  path is the residual-only `codeswitch_ckpt_v3` retrain. It requires the v2
  draft (`codeswitch_ckpt_v2`) or the restored latest `codeswitch_ckpt`; it
  copies the draft into v3 so the output is complete and evaluable.
* The full two-stage retrain (`DO_TRAIN = True`) is only a fallback. It rewrites
  `codeswitch_ckpt_v2`; do not enable it for this residual-only comparison.
* **Do not run the 60-utterance suite yet.** The defaults after the smoke cell
  are `DO_ARGMAN_EVAL = False`, `DO_SAMPLE_EVAL = False`, `DO_GATE = False`,
  and `DO_BUNDLE = False`. Listen to `output_audio/test_fixed_argmax.wav`
  first, then flip those four flags to `True` in the configuration cell.
* **B/C quality is not claimed until the gate cell passes.** If the gate fails,
  the notebook raises and leaves the claim blocked.


## 0. Configuration

Edit only this cell if your project/entity or branch differs.

In [ ]:
import os, sys, json, shutil, subprocess, filecmp, importlib.metadata as md
from pathlib import Path

REPO_URL = "https://github.com/tripathiji1312/codeSwitch.git"
REPO_DIR = Path("/kaggle/working/codeSwitch")
REPO_REF = "fix/013-bc-noise"       # set to "main" after the maintainer merges
WANDB_PROJECT = "codeswitch-tts"
WANDB_ENTITY = ""                   # e.g. "my-team"; leave empty for the default

CKPT_V1 = "codeswitch_ckpt"          # restored latest artifact
CKPT_V2 = "codeswitch_ckpt_v2"       # previous retrain (draft source for v3)
CKPT_V3 = "codeswitch_ckpt_v3"       # residual-only output (this run)
NUM_SENTENCES = 2000
MIN_TEACHER_SHARDS = 1500

DIAG_PRE = "audio_diag_pre"
DIAG_POST = "audio_diag_post"
PRIOR_SWEEP = "audio_diag_v3_prior"
ARG_DIR = "eval_results_2k_argmax"
SAMPLE_DIR = "eval_results_2k_sample"
ARG_AUDIO = "eval_audio_2k_argmax"
SAMPLE_AUDIO = "eval_audio_2k_sample"
ASR_MODEL = "openai/whisper-large-v3"
ZIP_PATH = "/kaggle/working/codeswitch_013_artifacts.zip"

SMOKE_UTT_ID = "ta-01"              # one held-out sentence; ta-01 is native Tamil
SMOKE_WAV = "output_audio/test_fixed_argmax.wav"

def _has_student(name):
    root = REPO_DIR / name
    return (root / "draft_cb0.pt").is_file() and (root / "residual_cb1_8.pt").is_file()

# Before the repo is cloned this falls back to CKPT_V1; the staged cells update
# it after training.  All post-train cells use EVAL_CKPT, never a hardcoded v2.
EVAL_CKPT = CKPT_V2 if _has_student(CKPT_V2) else CKPT_V1

# Toggle stages only when resuming a partially finished session.
DO_PRE_DIAG = True
DO_TRAIN = False                    # full two-stage fallback; off for residual-only v3
DO_RESIDUAL_RETRAIN = True          # primary path: v2 draft -> v3 residual
DO_PUSH_STUDENT = True
DO_POST_DIAG = True
DO_SMOKE = True
DO_PRIOR_SWEEP = True            # diagnostic only; no retraining
# Full evaluation is intentionally OFF until the smoke WAV is confirmed
# intelligible. Flip all four to True only after listening.
DO_ARGMAN_EVAL = False
DO_SAMPLE_EVAL = False
DO_GATE = False
DO_BUNDLE = False

def run(cmd, cwd=None, check=True, capture=False):
    """Run a subprocess with useful notebook output and fail loudly."""
    cmd = [str(c) for c in cmd]
    print("$", " ".join(cmd))
    p = subprocess.run(
        cmd, cwd=str(cwd) if cwd is not None else None,
        text=True, capture_output=capture,
    )
    if capture:
        if p.stdout:
            print(p.stdout)
        if p.stderr:
            print(p.stderr, file=sys.stderr)
    if check and p.returncode != 0:
        raise RuntimeError(f"command failed ({p.returncode}): {' '.join(cmd)}")
    return p

print("repo:", REPO_DIR)
print("branch:", REPO_REF)
print("initial eval checkpoint:", EVAL_CKPT)


## 1. W&B and Hugging Face secrets

The child processes inherit these environment variables, so
`bootstrap_diagnose.py`, `push_artifacts_to_wandb.py`, and the evaluation do not
need a separate `wandb.login()` call.

In [ ]:
def _secret(name):
    try:
        from kaggle_secrets import UserSecretsClient
        value = UserSecretsClient().get_secret(name)
        if value:
            return value
    except Exception as exc:
        print(f"[secrets] {name} not read from Kaggle Secrets ({type(exc).__name__})")
    return os.environ.get(name)

wandb_key = _secret("WANDB_API_KEY")
if not wandb_key:
    raise RuntimeError(
        "WANDB_API_KEY is missing. Add it in Kaggle Add-ons -> Secrets and enable Internet."
    )
os.environ["WANDB_API_KEY"] = wandb_key
print("W&B key: present (value hidden)")

hf_token = _secret("HF_TOKEN")
if hf_token:
    os.environ["HF_TOKEN"] = hf_token
    try:
        from huggingface_hub import login
        login(token=hf_token)
        print("Hugging Face: authenticated")
    except Exception as exc:
        print(f"Hugging Face login skipped: {type(exc).__name__}: {exc}")
else:
    print("Hugging Face: anonymous public downloads")


## 2. Sync the repository

This fetches the selected branch at its current remote commit. It does not
delete `teacher_cache/` or the restored checkpoints.

In [ ]:
if REPO_DIR.joinpath(".git").is_dir():
    try:
        run(["git", "fetch", "origin", REPO_REF], cwd=REPO_DIR)
    except RuntimeError as exc:
        raise RuntimeError(
            f"Could not fetch origin/{REPO_REF}. Push the branch first, or set "
            "REPO_REF='main' after merging. Original error: " + str(exc)
        )
    run(["git", "checkout", "-B", REPO_REF, "FETCH_HEAD"], cwd=REPO_DIR)
else:
    if REPO_DIR.exists():
        raise RuntimeError(
            f"{REPO_DIR} exists but is not a git checkout. Move it away or set REPO_DIR."
        )
    REPO_DIR.parent.mkdir(parents=True, exist_ok=True)
    run([
        "git", "clone", "--branch", REPO_REF, "--single-branch",
        REPO_URL, str(REPO_DIR),
    ], cwd=REPO_DIR.parent)

if not (REPO_DIR / "run_full_evaluation.py").is_file():
    raise RuntimeError(f"checkout is missing run_full_evaluation.py: {REPO_DIR}")

sys.path.insert(0, str(REPO_DIR))
commit = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO_DIR, text=True).strip()
print("commit:", commit)
print(subprocess.check_output(["git", "log", "--oneline", "-3"], cwd=REPO_DIR, text=True))


## 3. Dependency stack + W&B artifact restore

`bootstrap_diagnose.py` installs the same pinned stack as the old notebook
(`parler-tts`, `transformers==4.46.1`, `protobuf==5.29.5`, DAC, `wandb`, …),
then restores:

* `codeswitch-student:latest` → `codeswitch_ckpt/`
* `codeswitch-teacher-cache:latest` → `teacher_cache/` + corpus files

It stops after restore because this notebook drives the diagnostic separately.

**If this cell fails**, the usual causes are: Internet off, missing
`WANDB_API_KEY`, or the branch was not pushed. Read the `[bootstrap]` / `[W&B]`
lines before changing anything.

In [ ]:
restore_cmd = [
    sys.executable, "bootstrap_diagnose.py",
    "--repo", str(REPO_DIR),
    "--groups", "student,teacher",
    "--project", WANDB_PROJECT,
    "--skip-diagnose",
]
if WANDB_ENTITY:
    restore_cmd += ["--entity", WANDB_ENTITY]

restored = run(restore_cmd, cwd=REPO_DIR, check=False)
if restored.returncode != 0:
    raise RuntimeError(
        "Artifact restore failed. Check WANDB_API_KEY, Internet, and that the "
        "student/teacher artifacts exist. You can inspect with: "
        "python push_artifacts_to_wandb.py --download student --dest . "
        "(and the same for teacher)."
    )

for name in ("draft_cb0.pt", "residual_cb1_8.pt", "meta.json"):
    p = REPO_DIR / CKPT_V1 / name
    if not p.is_file():
        raise RuntimeError(f"restored v1 checkpoint is missing {p}")

shards = len(list((REPO_DIR / "teacher_cache").glob("shard_*.npz")))
print(f"teacher shards: {shards}")
if shards < MIN_TEACHER_SHARDS:
    raise RuntimeError(
        f"Only {shards} teacher shards restored; refusing to train on a partial/"
        "dry-run cache. Restore the 2 000-shard 'teacher' artifact instead of "
        "rerunning the teacher pass."
    )
if not (REPO_DIR / "codeswitch_corpus.txt").is_file():
    print("warning: codeswitch_corpus.txt is missing; --skip-teacher does not need it, "
          "but the teacher artifact normally includes it")


## 4. Preflight imports and branch check

This cell confirms that the checked-out branch really has the new
`--sample-mode` flag, the pinned stack imports cleanly, and the local Python
package is importable.

In [ ]:
import torch
import transformers
import wandb
from codeswitch_tts.utils.audio import DACWrapper
from codeswitch_tts.models.draft_transformer import DraftTransformer

print("python:", sys.version.split()[0])
print("torch:", torch.__version__, "| CUDA:", torch.cuda.is_available())
print("transformers:", transformers.__version__)
print("wandb:", wandb.__version__)
if not torch.cuda.is_available():
    raise RuntimeError("GPU is not available. Set the Kaggle accelerator to GPU T4 x1.")
if transformers.__version__ != "4.46.1":
    raise RuntimeError(
        f"transformers is {transformers.__version__}, expected 4.46.1. "
        "Run the restore cell again; do not mix package versions."
    )

help_text = subprocess.check_output(
    [sys.executable, "run_full_evaluation.py", "--help"], cwd=REPO_DIR, text=True
)
if "--sample-mode" not in help_text:
    raise RuntimeError(
        "The checked-out code does not expose --sample-mode. You are on an old "
        "commit; push/merge the fix branch or set REPO_REF correctly."
    )
print("B/C argmax flag: present")
print("teacher shards:", len(list((REPO_DIR / "teacher_cache").glob("shard_*.npz"))))


## 5. Pre-retrain diagnostic (v1)

This is evidence of the defect, not a quality claim. It writes code-level
top-1 percentages and valid DAC wavs to `audio_diag_pre/`. The
DAC backend must be `hf_pretrained` or `dac_official`; a synthetic/random
backend is refused.

In [ ]:
if DO_PRE_DIAG:
    diag_pre = run([
        sys.executable, "diagnose_audio_bug.py",
        "--ckpt", CKPT_V1,
        "--teacher-cache", "teacher_cache",
        "--out", DIAG_PRE,
        "--shards", "4",
        "--audio-frames", "172",
    ], cwd=REPO_DIR, check=False)
    if diag_pre.returncode != 0:
        raise RuntimeError("Pre-retrain diagnostic failed; see the lines above.")

    report = json.loads((REPO_DIR / DIAG_PRE / "diagnostic_report.json").read_text())
    summary = report["summary"]
    print("\npre-retrain backend:", report.get("dac_backend"))
    print("pre-retrain code metrics (%):")
    for key in ("cb0_s", "cb0_a", "rt_s", "rt_a", "rd_s", "rd_a"):
        print(f"  {key:5s} {summary[key]:.3f}")
    if report.get("dac_backend") not in ("hf_pretrained", "dac_official"):
        raise RuntimeError(
            "Diagnostic did not use real DAC weights; refusing to trust its wavs. "
            f"backend={report.get('dac_backend')!r}"
        )
    if not report.get("wavs"):
        raise RuntimeError("Diagnostic wrote no wavs; check the DAC load error above.")
else:
    print("pre-retrain diagnostic skipped")


## 6. Full two-stage retrain (fallback only)

`DO_TRAIN` is **False** by default. This cell is the old combined run: it
trains the draft and residual from the cached teacher shards and writes
`codeswitch_ckpt_v2`. Use it only when v2 is missing or the draft itself must
change; it is not the residual-only fix.

The primary path is the next cell: `--skip-draft --draft-ckpt codeswitch_ckpt_v2`
with the capacity-bumped residual, a 100-epoch ramp to full draft CB0, and
draft-CB0 checkpoint selection.

In [ ]:
if DO_TRAIN:
    train_cmd = [
        sys.executable, "train_draft_distill.py",
        "--corpus-json", "codeswitch_corpus.txt",
        "--num-sentences", str(NUM_SENTENCES),
        "--skip-teacher",
        "--draft-epochs", "300",
        "--residual-epochs", "300",
        "--residual-ramp-epochs", "100",
        "--residual-sched", "1.0",
        "--residual-d-model", "512",
        "--residual-layers", "6",
        "--min-epochs", "300",
        "--max-minutes", "60",
        "--max-minutes-residual", "90",
        "--teacher-cache", "teacher_cache",
        "--ckpt-dir", CKPT_V2,
    ]
    trained = run(train_cmd, cwd=REPO_DIR, check=False)
    if trained.returncode != 0:
        raise RuntimeError(
            "Full training failed. If the error mentions frames/codes, a teacher shard "
            "has a malformed target; inspect the named shard before retrying."
        )

    meta = json.loads((REPO_DIR / CKPT_V2 / "meta.json").read_text())
    print("\n=== retrained v2 meta ===")
    print(json.dumps(meta, indent=2)[:6000])
    draft_meta = meta.get("draft") or {}
    residual_meta = meta.get("residual") or {}
    print("\nsummary:")
    print("  draft epochs:", draft_meta.get("epochs_run"), "best:", draft_meta.get("best_epoch"))
    print("  draft val top-1:", draft_meta.get("val_top1"))
    print("  residual epochs:", residual_meta.get("epochs_run"), "best:", residual_meta.get("best_epoch"))
    print("  residual best_sched_p (selected):", residual_meta.get("best_sched_p"))
    print("  residual best_epoch_any/sched:",
          residual_meta.get("best_epoch_any"), "/", residual_meta.get("best_sched_p_any"))
    print("  residual final_sched_p:", residual_meta.get("final_sched_p"))
    print("  residual selected_full_draft:", residual_meta.get("selected_full_draft"))
    print("  residual cb_top1 teacher/draft:",
          residual_meta.get("cb_top1_teacher_mean"), "/", residual_meta.get("cb_top1_draft_mean"))
    if not draft_meta or not residual_meta:
        raise RuntimeError("meta.json is missing the draft or residual block.")
    if int(residual_meta.get("epochs_run") or 0) < 60:
        raise RuntimeError(
            "Residual trained for fewer than 60 epochs; it has not seen enough of "
            "the draft-CB0 distribution. Increase the residual wall-clock budget "
            "and retrain before evaluating."
        )
    if float(residual_meta.get("final_sched_p") or 0.0) < 0.95:
        raise RuntimeError(
            "Residual scheduled sampling did not reach the requested 1.0 target "
            "(final_sched_p < 0.95). The run stopped before the ramp completed; "
            "increase --max-minutes-residual and retrain before evaluating."
        )
    EVAL_CKPT = CKPT_V2
else:
    print("full two-stage training skipped")


## 6b. Residual-only v3 retrain from v2 — primary path

This is the authorized capacity + checkpoint-selection fix:

* `--skip-draft --draft-ckpt` loads the real v2 draft (not `None`), so
  scheduled sampling is active and the output v3 directory gets a complete
  draft checkpoint;
* `--residual-d-model 512 --residual-layers 6` widens the residual and adds
  two dilation blocks (receptive field ~253 frames);
* `--residual-ramp-epochs 100` ramps 0 → 100 % draft CB0 over the first 100
  epochs, then **holds** full draft for the remaining 50;
* validation and checkpoint selection use draft CB0, not teacher CB0;
* `--min-epochs 150` prevents a noisy early stop before the full-draft hold.

If the wall-clock cap stops before 150 epochs or `final_sched_p < 0.95`, the
cell raises. If `best_sched_p < 0.95`, it prints a warning: the selected state
was found before the full ramp, so inspect the diagnostic before trusting it.
No teacher pass is run (`--skip-teacher`), and v2 is left untouched.

In [ ]:
if DO_RESIDUAL_RETRAIN:
    draft_source = None
    for candidate in (CKPT_V2, CKPT_V1):
        if (REPO_DIR / candidate / "draft_cb0.pt").is_file():
            draft_source = candidate
            break
    if draft_source is None:
        raise RuntimeError(
            "No draft checkpoint found. Expected codeswitch_ckpt_v2 (or the restored "
            "latest codeswitch_ckpt). Run the full two-stage cell or restore v2 first."
        )
    if draft_source != CKPT_V2:
        print(f"warning: {CKPT_V2} is absent; using restored latest artifact {draft_source} "
              "as the draft source. Verify its meta.json is the v2 draft.")

    source_meta_path = REPO_DIR / draft_source / "meta.json"
    if source_meta_path.is_file():
        source_meta = json.loads(source_meta_path.read_text())
        src_draft = source_meta.get("draft") or {}
        print("draft source:", draft_source,
              "| best_epoch:", src_draft.get("best_epoch"),
              "| val_top1:", src_draft.get("val_top1"))

    residual_cmd = [
        sys.executable, "train_draft_distill.py",
        "--corpus-json", "codeswitch_corpus.txt",
        "--num-sentences", str(NUM_SENTENCES),
        "--skip-teacher",
        "--skip-draft",
        "--draft-ckpt", draft_source,
        "--residual-epochs", "150",
        "--residual-ramp-epochs", "100",
        "--residual-sched", "1.0",
        "--residual-d-model", "512",
        "--residual-layers", "6",
        "--min-epochs", "150",
        "--max-minutes-residual", "90",
        "--teacher-cache", "teacher_cache",
        "--ckpt-dir", CKPT_V3,
    ]
    trained = run(residual_cmd, cwd=REPO_DIR, check=False)
    if trained.returncode != 0:
        raise RuntimeError(
            "Residual-only v3 training failed. Inspect the draft-ckpt path and the "
            "residual log above; do not fall back to a silent teacher-CB0 run."
        )

    v3_dir = REPO_DIR / CKPT_V3
    v3_meta_path = v3_dir / "meta.json"
    if not v3_meta_path.is_file():
        raise RuntimeError(f"v3 wrote no meta.json: {v3_meta_path}")
    meta = json.loads(v3_meta_path.read_text())
    residual_meta = meta.get("residual") or {}
    draft_meta = meta.get("draft") or {}
    if not residual_meta or not draft_meta:
        raise RuntimeError("v3 meta.json is missing the draft or residual block.")
    if not (v3_dir / "residual_cb1_8.pt").is_file():
        raise RuntimeError("v3 is missing residual_cb1_8.pt")
    copied_draft = v3_dir / "draft_cb0.pt"
    source_draft = REPO_DIR / draft_source / "draft_cb0.pt"
    if not copied_draft.is_file() or not filecmp.cmp(copied_draft, source_draft, shallow=False):
        raise RuntimeError("v3 draft_cb0.pt is missing or differs from the source draft")
    if not meta.get("residual_only"):
        raise RuntimeError("v3 meta does not record residual_only=true")
    if float(meta.get("residual_scheduled_sampling") or 0.0) < 1.0:
        raise RuntimeError("v3 did not request full draft-CB0 scheduled sampling")

    print("\n=== residual-only v3 meta ===")
    print(json.dumps(meta, indent=2)[:8000])
    print("\nsummary:")
    print("  residual epochs:", residual_meta.get("epochs_run"),
          "best:", residual_meta.get("best_epoch"))
    print("  residual capacity:", meta.get("residual_capacity"))
    print("  residual best_sched_p (selected):", residual_meta.get("best_sched_p"))
    print("  residual best_epoch_any/sched:",
          residual_meta.get("best_epoch_any"), "/", residual_meta.get("best_sched_p_any"))
    print("  residual final_sched_p:", residual_meta.get("final_sched_p"))
    print("  residual selected_full_draft:", residual_meta.get("selected_full_draft"))
    print("  residual selection:", residual_meta.get("selection_metric"))
    print("  residual cb_top1 teacher/draft:",
          residual_meta.get("cb_top1_teacher_mean"), "/",
          residual_meta.get("cb_top1_draft_mean"))

    if int(residual_meta.get("epochs_run") or 0) < 150:
        raise RuntimeError(
            "Residual trained for fewer than 150 epochs; the final full-draft hold "
            "did not complete. Increase --max-minutes-residual and retrain."
        )
    if float(residual_meta.get("final_sched_p") or 0.0) < 0.95:
        raise RuntimeError(
            "Residual did not reach the full draft-CB0 schedule "
            "(final_sched_p < 0.95); inspect the ramp and retrain."
        )
    if not residual_meta.get("selected_full_draft"):
        raise RuntimeError(
            "No full-draft checkpoint was selected (selected_full_draft is false). "
            f"best_epoch={residual_meta.get('best_epoch')} "
            f"best_sched_p={residual_meta.get('best_sched_p')} "
            f"final_sched_p={residual_meta.get('final_sched_p')}. The run stopped "
            "before the full-draft hold; increase --max-minutes-residual and retrain."
        )
    EVAL_CKPT = CKPT_V3
    print("residual-only v3 ready; EVAL_CKPT =", EVAL_CKPT)
else:
    print("residual-only v3 training skipped")

## 7. Push the residual-only v3 student to W&B

This writes a new `codeswitch-student` artifact version. The artifact payload is
always named `codeswitch_ckpt` inside W&B, even when the local directory is
`codeswitch_ckpt_v3`; that is expected. This cell is the recovery point for a
session death after training and before evaluation.

In [ ]:
if DO_PUSH_STUDENT:
    student_tag = "student-v3" if EVAL_CKPT == CKPT_V3 else "student-v2"
    push_cmd = [
        sys.executable, "push_artifacts_to_wandb.py",
        "--group", "student",
        "--ckpt-dir", EVAL_CKPT,
        "--project", WANDB_PROJECT,
        "--tags", f"kaggle,plan013,{student_tag}",
        "--strict",
    ]
    if WANDB_ENTITY:
        push_cmd += ["--entity", WANDB_ENTITY]
    pushed = run(push_cmd, cwd=REPO_DIR, check=False)
    if pushed.returncode != 0:
        raise RuntimeError("W&B student backup failed; keep the session alive and retry.")
else:
    print("student push skipped")


## 8. Post-retrain diagnostic + code gate (v3)

This is the Phase-4 gate. It must report **both**:

* `rt_a >= 5 %` — residual works on teacher CB0;
* `rd_a >= 5 %` — residual works on the draft CB0 it will actually see.

If either fails, the notebook raises and no B/C quality claim is made.

In [ ]:
if DO_POST_DIAG:
    diag_post = run([
        sys.executable, "diagnose_audio_bug.py",
        "--ckpt", EVAL_CKPT,
        "--teacher-cache", "teacher_cache",
        "--out", DIAG_POST,
        "--shards", "4",
        "--audio-frames", "172",
    ], cwd=REPO_DIR, check=False)
    if diag_post.returncode != 0:
        raise RuntimeError("Post-retrain diagnostic failed; see the lines above.")

    report = json.loads((REPO_DIR / DIAG_POST / "diagnostic_report.json").read_text())
    summary = report["summary"]
    print("\npost-retrain backend:", report.get("dac_backend"))
    print("post-retrain code metrics (%):")
    for key in ("cb0_s", "cb0_a", "rt_s", "rt_a", "rd_s", "rd_a"):
        print(f"  {key:5s} {summary[key]:.3f}")

    failures = []
    if report.get("dac_backend") not in ("hf_pretrained", "dac_official"):
        failures.append(f"DAC backend is {report.get('dac_backend')!r}")
    if float(summary["rt_a"]) < 5.0:
        failures.append(f"rt_a={summary['rt_a']:.3f}% < 5%")
    if float(summary["rd_a"]) < 5.0:
        failures.append(f"rd_a={summary['rd_a']:.3f}% < 5%")
    ((REPO_DIR / DIAG_POST) / "code_gate.json").write_text(json.dumps({
        "summary": summary, "dac_backend": report.get("dac_backend"),
        "failures": failures,
    }, indent=2))
    if failures:
        raise AssertionError("POST-RETRAIN CODE GATE FAILED: " + "; ".join(failures))
    print("\nPOST-RETRAIN CODE GATE PASSED (rt_a and rd_a >= 5%)")
else:
    print("post-retrain diagnostic skipped")


## 9. Immediate one-sentence argmax smoke test

**Do not run the 60-utterance suite yet.** This cell synthesizes one held-out
sentence (`ta-01`) with `--sample-mode argmax`, saves
`output_audio/test_fixed_argmax.wav`, and prints:

* draft CB0 top-1 accuracy against a **fresh Parler teacher target for that
  exact sentence** (not a training-cache shard);
* residual CB1–8 top-1 on teacher CB0 and on draft CB0;
* the DAC backend and basic audio statistics.

The script refuses to write a WAV through a synthetic/random DAC. Listen to the
WAV below. Only after it sounds like intelligible speech should you set
`DO_ARGMAN_EVAL = True`, `DO_SAMPLE_EVAL = True`, `DO_GATE = True`, and
`DO_BUNDLE = True` in the configuration cell, then run the remaining cells.


In [ ]:
if DO_SMOKE:
    smoke_wav = REPO_DIR / SMOKE_WAV
    smoke_json = smoke_wav.with_suffix(".json")
    smoke_cmd = [
        sys.executable, "smoke_test_argmax.py",
        "--ckpt", EVAL_CKPT,
        "--utt-id", SMOKE_UTT_ID,
        "--config", "C",
        "--out", SMOKE_WAV,
        "--seed", "42",
        "--device", "cuda:0",
    ]
    smoke = run(smoke_cmd, cwd=REPO_DIR, check=False)
    if smoke.returncode != 0:
        raise RuntimeError(
            "ONE-SENTENCE SMOKE TEST FAILED. Do not launch the 60-utterance run; "
            "inspect the DAC/Parler/top-1 output above."
        )
    if not smoke_json.is_file():
        raise RuntimeError(f"smoke test wrote no report: {smoke_json}")
    smoke_report = json.loads(smoke_json.read_text())
    print("\nSMOKE SUMMARY")
    print("  utterance       :", smoke_report.get("utt_id"))
    print("  CB0 top-1       :", smoke_report.get("top1_cb0_pct"))
    print("  residual teacher:", smoke_report.get("residual_teacher_cb1_8_top1_pct"))
    print("  residual draft  :", smoke_report.get("residual_draft_cb1_8_top1_pct"))
    print("  DAC backend     :", smoke_report.get("dac_backend"))
    print("  WAV             :", smoke_wav)

    from IPython.display import Audio, display
    display(Audio(str(smoke_wav)))
else:
    print("one-sentence smoke test skipped")


## 10. Listen to the diagnostic wavs

These are the valid DAC outputs, not the old random-weight wavs. Listen to
`teacher_full` as the reference, then the isolation pairs:

* `student_cb0_argmax_teacher_res` vs `student_cb0_teacher_res_sample`:
  draft CB0 argmax vs sample, with the teacher CB1-8 held fixed.
* `teacher_cb0_student_res_argmax` vs `teacher_cb0_student_res_sample`:
  residual argmax vs sample, with the teacher CB0 held fixed.
* `student_cb0_argmax_res_sample` and `student_cb0_sample_res_argmax`:
  cross-checks that change one side at a time.
* `student_full_argmax` and `student_full_sample`: the deployed paths.

The `[code]` lines print draft CB0 uniqueness, top-token fraction, longest run
and entropy. A high top-token fraction with a long run is argmax mode collapse;
that is a draft failure, not a residual failure.


In [ ]:
from IPython.display import Audio, display
import glob

diag_dir = REPO_DIR / DIAG_POST
report_path = diag_dir / "diagnostic_report.json"
if report_path.is_file():
    report = json.loads(report_path.read_text())
    code_summary = report.get("code_summary") or {}
    if code_summary:
        print("draft CB0 code stats (sample -> argmax):")
        for key in ("unique", "top_frac", "longest_run", "entropy_nats"):
            print(f"  {key:12s} {code_summary[f'draft_sample_{key}']:.3f}"
                  f" -> {code_summary[f'draft_argmax_{key}']:.3f}")

wavs = sorted(glob.glob(str(diag_dir / "shard_*_teacher_full.wav")))
print(f"{len(wavs)} teacher reference wav(s) in {diag_dir}")
for p in wavs[:2]:
    prefix = p.replace("_teacher_full.wav", "")
    for tag in ("teacher_full", "student_cb0_teacher_res_sample",
                "student_cb0_argmax_teacher_res",
                "teacher_cb0_student_res_sample",
                "teacher_cb0_student_res_argmax",
                "student_cb0_argmax_res_sample",
                "student_cb0_sample_res_argmax",
                "student_cb0_only_sample", "student_cb0_only_argmax",
                "student_full_argmax", "student_full_sample"):
        candidate = f"{prefix}_{tag}.wav"
        if Path(candidate).is_file():
            print(f"\n{tag}: {Path(candidate).name}")
            display(Audio(candidate))


## 10b. Draft-prior argmax sweep (diagnostic, no retrain)

The v3 draft's argmax collapses to one token (`top_frac ≈ 0.999`, longest run
= whole utterance) while its sampled sequence is diverse. This cell tests
whether that is a dominant global token prior by subtracting
`alpha * log p_teacher(token)` from the draft logits. It does **not** modify
the checkpoint. Listen to the `teacher_res` WAVs: they hold the teacher CB1-8
fixed and change only the adjusted draft CB0. If no alpha gives a varied,
intelligible CB0, the next step is a draft retrain, not another residual run.


In [ ]:
if DO_PRIOR_SWEEP:
    sweep_dir = REPO_DIR / PRIOR_SWEEP
    sweep_cmd = [
        sys.executable, "tune_draft_prior.py",
        "--ckpt", EVAL_CKPT,
        "--teacher-cache", "teacher_cache",
        "--out", PRIOR_SWEEP,
        "--shards", "4",
        "--alphas", "0,0.25,0.5,0.75,1.0",
        "--audio-alphas", "0,0.5,1.0",
        "--device", "cuda:0",
    ]
    swept = run(sweep_cmd, cwd=REPO_DIR, check=False)
    if swept.returncode != 0:
        raise RuntimeError("Draft-prior sweep failed; inspect the log above.")

    report = json.loads((sweep_dir / "prior_sweep.json").read_text())
    print("\nPRIOR SWEEP SUMMARY")
    print("alpha  cb0_top1  rd_a    unique  top_frac  run    H")
    for alpha in report["alphas"]:
        s = report["summary"][f"{alpha:.2f}"]
        print(f"{alpha:5.2f}  {s['cb0_top1']:7.3f}%  {s['rd_a']:6.3f}%  "
              f"{s['unique']:6.0f}  {s['top_frac']:8.3f}  {s['longest_run']:5.0f}  "
              f"{s['entropy_nats']:5.2f}")

    from IPython.display import Audio, display
    shown = 0
    for w in report.get("wavs", []):
        if w.get("tag") == "teacher_res" and float(w.get("alpha", -1)) in (0.5, 1.0):
            print(f"\nshard {w['shard']} alpha {w['alpha']:.2f} {w['tag']}: {Path(w['file']).name}")
            display(Audio(w["file"]))
            shown += 1
            if shown >= 4:
                break
else:
    print("draft-prior sweep skipped")


## 11. Full argmax evaluation — A, B, C

`--sample-mode argmax` makes both the draft CB0 and the residual CB1–8 greedy.
`--figures-dir` is set to the same directory as `--results` so the gate can read
`<results>/summary.json` directly.

This is the main quality run. It is resumable: rerunning the cell skips rows
already present unless `--force` is supplied.

In [ ]:
if DO_ARGMAN_EVAL:
    arg_cmd = [
        sys.executable, "run_full_evaluation.py",
        "--ckpt", EVAL_CKPT,
        "--configs", "A,B,C",
        "--sample-mode", "argmax",
        "--results", f"{ARG_DIR}/raw_results.jsonl",
        "--audio-dir", ARG_AUDIO,
        "--figures-dir", ARG_DIR,
        "--asr-model", ASR_MODEL,
        "--force",
    ]
    ev = run(arg_cmd, cwd=REPO_DIR, check=False)
    if ev.returncode != 0:
        raise RuntimeError(
            "Argmax evaluation failed. Do not fall back to synthetic/untrained "
            "audio; fix the Parler/DAC/ASR stack first."
        )
    summary_path = REPO_DIR / ARG_DIR / "summary.json"
    if not summary_path.is_file():
        raise RuntimeError(f"missing evaluation summary: {summary_path}")

    # The plan gate also requires 60 clean rows per config and row provenance.
    rows = [json.loads(line) for line in (REPO_DIR / ARG_DIR / "raw_results.jsonl").read_text().splitlines() if line.strip()]
    errors = [r for r in rows if r.get("error")]
    if errors:
        raise RuntimeError(f"argmax evaluation wrote {len(errors)} error row(s); inspect the log")
    allowed_backends = {"parler_streamer", "parler_audio_encoder", "hf_pretrained",
                        "dac_official", "passed_model"}
    for cfg in ("A", "B", "C"):
        sub = [r for r in rows if r.get("config") == cfg]
        if len(sub) < 60:
            raise RuntimeError(f"argmax {cfg} has only {len(sub)} rows; expected 60")
        if cfg in ("B", "C"):
            bad_mode = [r for r in sub if r.get("sample_mode") != "argmax"]
            if bad_mode:
                raise RuntimeError(f"argmax {cfg} has {len(bad_mode)} row(s) with the wrong sample_mode")
            bad_backend = [r for r in sub if r.get("dac_backend") not in allowed_backends]
            if bad_backend:
                raise RuntimeError(f"argmax {cfg} has {len(bad_backend)} row(s) with a non-real DAC backend")
    print("argmax rows:", len(rows), "backend/sample_mode provenance: OK")
    print("argmax summary:", summary_path)
else:
    print("argmax evaluation skipped")


## 12. Matched sample evaluation — B, C

This is the latency/comparison baseline. It is not used to claim the fixed
quality; it exists so the argmax latency claims can be compared to the old
sampling behaviour on the same checkpoint.

In [ ]:
if DO_SAMPLE_EVAL:
    sample_cmd = [
        sys.executable, "run_full_evaluation.py",
        "--ckpt", EVAL_CKPT,
        "--configs", "B,C",
        "--sample-mode", "sample",
        "--results", f"{SAMPLE_DIR}/raw_results.jsonl",
        "--audio-dir", SAMPLE_AUDIO,
        "--figures-dir", SAMPLE_DIR,
        "--asr-model", ASR_MODEL,
        "--force",
    ]
    ev = run(sample_cmd, cwd=REPO_DIR, check=False)
    if ev.returncode != 0:
        raise RuntimeError("Matched sample evaluation failed; see the lines above.")
    summary_path = REPO_DIR / SAMPLE_DIR / "summary.json"
    if not summary_path.is_file():
        raise RuntimeError(f"missing evaluation summary: {summary_path}")

    rows = [json.loads(line) for line in (REPO_DIR / SAMPLE_DIR / "raw_results.jsonl").read_text().splitlines() if line.strip()]
    errors = [r for r in rows if r.get("error")]
    if errors:
        raise RuntimeError(f"sample evaluation wrote {len(errors)} error row(s); inspect the log")
    allowed_backends = {"parler_streamer", "parler_audio_encoder", "hf_pretrained",
                        "dac_official", "passed_model"}
    for cfg in ("B", "C"):
        sub = [r for r in rows if r.get("config") == cfg]
        if len(sub) < 60:
            raise RuntimeError(f"sample {cfg} has only {len(sub)} rows; expected 60")
        bad_mode = [r for r in sub if r.get("sample_mode") != "sample"]
        if bad_mode:
            raise RuntimeError(f"sample {cfg} has {len(bad_mode)} row(s) with the wrong sample_mode")
        bad_backend = [r for r in sub if r.get("dac_backend") not in allowed_backends]
        if bad_backend:
            raise RuntimeError(f"sample {cfg} has {len(bad_backend)} row(s) with a non-real DAC backend")
    print("sample rows:", len(rows), "backend/sample_mode provenance: OK")
    print("sample summary:", summary_path)
else:
    print("sample evaluation skipped")


## 13. Decision gate

This mirrors the plan-013 gate. CER is in percentage points, using
`cer_latin.p50` when present, otherwise `cer.p50`. The gate checks:

* all three argmax configs have rows;
* `B_CER <= A_CER + 10 pp`, `C_CER <= B_CER + 10 pp`;
* `B_UTMOS >= 0.75 * A_UTMOS`;
* no ASR loops (`asr_fail == 0` for A/B/C);
* B/C `ttfa_compute_warm.p50` and `rtf.p50` within ±25 % of the sample run.

If it fails, do **not** loosen the thresholds and do not claim B/C quality.

In [ ]:
if DO_GATE:
    def _load_agg(directory):
        path = REPO_DIR / directory / "summary.json"
        if not path.is_file():
            raise RuntimeError(f"missing summary: {path}")
        return json.loads(path.read_text())["aggregates"]

    arg = _load_agg(ARG_DIR)
    samp = _load_agg(SAMPLE_DIR)
    failures = []

    def check(ok, msg):
        print(("PASS  " if ok else "FAIL  ") + msg)
        if not ok:
            failures.append(msg)

    def cer_p50(a):
        vals = (a.get("values") or {}).get("cer_latin") or []
        key = "cer_latin" if any(v is not None for v in vals) else "cer"
        v = (a.get(key) or {}).get("p50")
        return None if v is None else float(v)

    for cfg in ("A", "B", "C"):
        check((arg.get(cfg) or {}).get("n", 0) > 0, f"argmax {cfg} has rows")

    a_cer, b_cer, c_cer = (cer_p50(arg.get(k) or {}) for k in ("A", "B", "C"))
    check(a_cer is not None and b_cer is not None, "CER p50 present for A and B")
    check(c_cer is not None, "CER p50 present for C")
    if a_cer is not None and b_cer is not None:
        check(100 * b_cer <= 100 * a_cer + 10,
              f"B CER p50 {100*b_cer:.1f}% <= A {100*a_cer:.1f}% + 10 pp")
    if b_cer is not None and c_cer is not None:
        check(100 * c_cer <= 100 * b_cer + 10,
              f"C CER p50 {100*c_cer:.1f}% <= B {100*b_cer:.1f}% + 10 pp")

    a_u = ((arg.get("A") or {}).get("utmos") or {}).get("mean")
    b_u = ((arg.get("B") or {}).get("utmos") or {}).get("mean")
    check(a_u is not None and b_u is not None, "UTMOS mean present for A and B")
    if a_u is not None and b_u is not None:
        check(float(b_u) >= 0.75 * float(a_u),
              f"B UTMOS {float(b_u):.3f} >= 0.75 * A {float(a_u):.3f}")

    for cfg in ("A", "B", "C"):
        check(int((arg.get(cfg) or {}).get("asr_fail", -1)) == 0,
              f"{cfg} asr_fail == 0 (CER <= 2.0)")

    for cfg in ("B", "C"):
        for key, label in (("ttfa_compute_warm", "TTFA-compute-warm p50"),
                           ("rtf", "RTF p50")):
            x = ((arg.get(cfg) or {}).get(key) or {}).get("p50")
            y = ((samp.get(cfg) or {}).get(key) or {}).get("p50")
            ok = x is not None and y not in (None, 0) and abs(float(x) / float(y) - 1) <= 0.25
            check(ok, f"{cfg} {label} {x} vs sample {y} within +/-25%")

    gate_report = {"failures": failures, "argmax": arg, "sample": samp}
    (REPO_DIR / ARG_DIR / "gate_report.json").write_text(
        json.dumps(gate_report, indent=2, default=str)
    )
    print(f"\n{len(failures)} gate failure(s)")
    if failures:
        raise AssertionError("PLAN-013 QUALITY GATE FAILED: " + "; ".join(failures))
    print("PLAN-013 QUALITY GATE PASSED")
else:
    print("quality gate skipped")


## 14. Zip and back up the run

The zip excludes `.pt` weights (they are already in the `student` artifact). It
contains both evaluation runs, both diagnostics, and the retrained `meta.json`.
The final W&B `bundle` artifact is the off-Kaggle copy.

In [ ]:
if DO_BUNDLE:
    entries = [
        SMOKE_WAV, f"{Path(SMOKE_WAV).with_suffix('.json')}",
        ARG_DIR, SAMPLE_DIR, ARG_AUDIO, SAMPLE_AUDIO,
        DIAG_PRE, DIAG_POST, f"{EVAL_CKPT}/meta.json",
    ]
    existing = [e for e in entries if (REPO_DIR / e).exists()]
    if not existing:
        raise RuntimeError("nothing to zip; run the evaluation/diagnostic cells first")
    zip_path = Path(ZIP_PATH)
    zip_path.unlink(missing_ok=True)
    zipped = run(["zip", "-qr", str(zip_path), *existing, "-x", "*.pt"],
                 cwd=REPO_DIR, check=False)
    if zipped.returncode != 0:
        raise RuntimeError("zip failed; inspect the paths above")
    print("zip:", zip_path, f"{zip_path.stat().st_size / 1e6:.1f} MB")

    bundle_cmd = [
        sys.executable, "push_artifacts_to_wandb.py",
        "--group", "bundle",
        "--zip", str(zip_path),
        "--project", WANDB_PROJECT,
        "--tags", "kaggle,plan013,bundle",
        "--strict",
    ]
    if WANDB_ENTITY:
        bundle_cmd += ["--entity", WANDB_ENTITY]
    bundled = run(bundle_cmd, cwd=REPO_DIR, check=False)
    if bundled.returncode != 0:
        raise RuntimeError("W&B bundle backup failed; keep the session alive and retry.")
else:
    print("bundle skipped")


## 15. What to send back

At minimum:

* `output_audio/test_fixed_argmax.wav` and `.json` (the listening check)
* `codeswitch_ckpt_v3/meta.json` (or the current `EVAL_CKPT`)
* `eval_results_2k_argmax/summary.json`, `raw_results.jsonl`, `gate_report.json`
* `eval_results_2k_sample/summary.json`
* `audio_diag_post/diagnostic_report.json` and the `student_full_*` / `teacher_full` wavs
* `/kaggle/working/codeswitch_013_artifacts.zip`

If the gate failed, send the failure list and the diagnostic `rt_a`/`rd_a`
values. Do not report B/C as fixed until the gate passes.
